# Lab 07 — Data Leakage & Validation Design

*AI Cybersecurity Masterclass · Lab 07 · AI60secLab*

## Research question
Is the model learning real attack patterns, or an accidental shortcut?

Labs 01–06 produced scores. This lab asks whether those scores are trustworthy. Leakage is when the model sees information it would not have in production — the score goes up, and the model gets worse in the real world.

## Learning objectives
- confirm the answer is not hiding inside the feature matrix;
- fit preprocessing on training data only, and see what happens when you don't;
- wrap preprocessing and the model in one `Pipeline` so leakage is hard to cause by accident;
- check for duplicate rows shared across train and test;
- run a shuffled-label sanity test that a real result must fail;
- document the validation design so a reader can judge whether to believe you.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns, build_preprocessor
from src.features import drop_identifier_like_columns
from src.evaluation import binary_metrics, binary_metrics_frame

train, test = load_unsw(DATA_DIR)

X_train, y_train = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)

## Step 1 — Verify the label columns are gone from X

The first and most embarrassing form of leakage is training on the answer. `split_xy` drops both `label` and `attack_cat`, but never assume — assert it.

`attack_cat` matters as much as `label` here: it encodes the answer to the binary question too. If `attack_cat` is anything other than "Normal", the label is 1.

In [ ]:
forbidden = ["label", "attack_cat"]

for name, frame in [("X_train", X_train), ("X_test", X_test)]:
    leaked = [c for c in forbidden if c in frame.columns]
    print(f"{name}: leaked target columns -> {leaked if leaked else 'none'}")
    assert not leaked, f"{name} still contains a target column"

# A feature perfectly correlated with the target is also suspicious.
numeric = X_train.select_dtypes(include="number")
correlations = numeric.corrwith(y_train).abs().sort_values(ascending=False)

print("\nMost target-correlated features (|r| > 0.9 deserves investigation):")
print(correlations.head(8).round(3).to_string())

## Step 2 — Fit preprocessing on the training data only

This is the pattern from the lesson. The important line is the `fit` on `X_train` — the preprocessor learns categories and scaling parameters **only** from training data, then applies those learned rules to both sets.

In [ ]:
preprocessor = build_preprocessor(X_train, scale_numeric=False)
model = RandomForestClassifier(
    n_estimators=250, class_weight="balanced_subsample",
    random_state=42, n_jobs=-1,
)

preprocessor.fit(X_train)

X_train_ready = preprocessor.transform(X_train)
X_test_ready  = preprocessor.transform(X_test)

model.fit(X_train_ready, y_train)
y_pred = model.predict(X_test_ready)

honest = binary_metrics(y_test, y_pred)
display(binary_metrics_frame("train-only preprocessing", honest).round(4))

## Step 3 — The dangerous shortcut

Now do it wrong on purpose. Fit the preprocessor on train and test combined — the move that feels harmless because "it's only preprocessing, not the model."

The test distribution has now influenced how training data was prepared. Compare the two scores: on this dataset the gap is usually small, and **that is exactly why leakage is dangerous.** It rarely announces itself with an obviously fake number. It quietly borrows a little information, and you never find out until production.

In [ ]:
leaky_preprocessor = build_preprocessor(X_train, scale_numeric=False)

combined = pd.concat([X_train, X_test], axis=0)
leaky_preprocessor.fit(combined)          # <-- the mistake: test data shaped the transform

leaky_model = RandomForestClassifier(
    n_estimators=250, class_weight="balanced_subsample",
    random_state=42, n_jobs=-1,
).fit(leaky_preprocessor.transform(X_train), y_train)

leaky_pred = leaky_model.predict(leaky_preprocessor.transform(X_test))
leaked = binary_metrics(y_test, leaky_pred)

display(pd.concat([
    binary_metrics_frame("train-only (honest)", honest),
    binary_metrics_frame("fit on train+test (leaky)", leaked),
]).round(4))

print(f"F1 difference: {leaked.f1 - honest.f1:+.4f}")
print("A small gap does not mean the shortcut is safe — it means leakage is hard to detect by score alone.")

## Step 4 — Use a scikit-learn Pipeline

The Pipeline links preprocessing and the classifier into one ordered object. Call `fit` and preprocessing is learned from the training data only, then flows into the model. Call `predict` and the same learned transform is applied to new data.

You cannot accidentally fit on the test set, because you never touch the preprocessor yourself.

In [ ]:
classifier = RandomForestClassifier(
    n_estimators=250, class_weight="balanced_subsample",
    random_state=42, n_jobs=-1,
)

from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('prep', preprocessor),
    ('model', classifier)
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

pipeline_metrics = binary_metrics(y_test, y_pred)
display(binary_metrics_frame("Pipeline", pipeline_metrics).round(4))

print("Matches the manual train-only result:",
      np.isclose(pipeline_metrics.f1, honest.f1, atol=1e-9))
print("Same discipline, fewer ways to get it wrong.")

## Step 5 — Check for duplicate rows across the split

If a near-identical row appears in both train and test, the model has effectively seen the answer. Test performance goes up for no honest reason.

In [ ]:
train_hash = X_train.astype(str).agg('|'.join, axis=1)
test_hash  = X_test.astype(str).agg('|'.join, axis=1)

overlap = set(train_hash).intersection(set(test_hash))
print('Duplicate overlap:', len(overlap))

# How many test ROWS does that overlap actually account for?
overlapping_rows = test_hash.isin(overlap)
n_overlap = int(overlapping_rows.sum())

print(f"Duplicate rows inside train : {train_hash.duplicated().sum()}")
print(f"Duplicate rows inside test  : {test_hash.duplicated().sum()}")
print(f"Test rows also seen in train: {n_overlap}  ({n_overlap / len(X_test):.2%} of the test set)")

## Step 6 — Related entities crossing the split

Exact duplicates are the easy case. The harder one is *related* rows: two records belonging to the same underlying entity, one in train and one in test. The model memorizes the entity instead of learning the behaviour.

UNSW-NB15's prepared partition ships no session or host identifier, so we cannot group by entity here — and saying that out loud is part of the audit. The check below approximates it by ignoring the most volatile numeric fields and looking for rows that are otherwise identical across the split.

In [ ]:
volatile = [c for c in ["dur", "rate", "sbytes", "dbytes", "spkts", "dpkts"]
            if c in X_train.columns]
stable_cols = [c for c in X_train.columns if c not in volatile]

train_stable = X_train[stable_cols].astype(str).agg('|'.join, axis=1)
test_stable  = X_test[stable_cols].astype(str).agg('|'.join, axis=1)

near_overlap = set(train_stable).intersection(set(test_stable))
n_near = int(test_stable.isin(near_overlap).sum())

print(f"Test rows sharing a stable-field signature with train: {n_near} "
      f"({n_near / len(X_test):.2%})")
print()
print("This is a proxy, not proof. Record the limitation explicitly:")
print("no session/host identifier exists, so entity-level leakage cannot be ruled out.")

## Step 7 — Correct what you found, then re-score

If a check found something, fix it and recalculate. A number that moves after a correction is evidence. A number you never re-ran is just a claim.

In [ ]:
if n_overlap:
    clean = ~overlapping_rows.values
    corrected = binary_metrics(
        y_test[clean],
        pipe.predict(X_test[clean]),
        pipe.predict_proba(X_test[clean])[:, 1],
    )
    display(pd.concat([
        binary_metrics_frame("as reported (full test set)", pipeline_metrics),
        binary_metrics_frame("corrected (duplicates removed)", corrected),
    ]).round(4))
    print(f"F1 moved {corrected.f1 - pipeline_metrics.f1:+.4f} once duplicates were removed.")
    reported = corrected
else:
    print("No exact overlap to correct — report the check as performed and passed.")
    print("A negative result is still evidence, provided you actually ran it.")
    reported = pipeline_metrics

## Step 8 — Try to break your own result

Shuffle the labels and retrain. If the data carries real signal, performance should **collapse to roughly chance**. If a shuffled-label model still scores well, something is leaking structure — duplicated rows, an index-like feature, or a target-derived column.

This is the cheapest credibility check in the whole course, and the one reviewers respect most.

In [ ]:
rng = np.random.default_rng(42)
y_shuffled = pd.Series(rng.permutation(y_train.values), index=y_train.index)

shuffled_pipe = Pipeline([
    ("prep", build_preprocessor(X_train, scale_numeric=False)),
    ("model", RandomForestClassifier(
        n_estimators=150, class_weight="balanced_subsample",
        random_state=42, n_jobs=-1)),
]).fit(X_train, y_shuffled)

shuffled_pred  = shuffled_pipe.predict(X_test)
shuffled_score = shuffled_pipe.predict_proba(X_test)[:, 1]
shuffled_metrics = binary_metrics(y_test, shuffled_pred, shuffled_score)

real_score = pipe.predict_proba(X_test)[:, 1]
real_metrics = binary_metrics(y_test, y_pred, real_score)

display(pd.concat([
    binary_metrics_frame("real labels", real_metrics),
    binary_metrics_frame("shuffled labels", shuffled_metrics),
]).round(4))

# Judge this on ROC AUC, not F1. With an imbalanced base rate a useless model can
# still post a respectable F1 simply by favouring the majority class. AUC has no
# such escape hatch: destroy the labels and it must fall to ~0.50.
print(f"Attack base rate in test : {y_test.mean():.3f}")
print(f"Real-label AUC           : {real_metrics.roc_auc:.3f}")
print(f"Shuffled-label AUC       : {shuffled_metrics.roc_auc:.3f}  (0.50 = chance)")

print("\nPASS — shuffled labels collapse to chance, so the signal is real."
      if shuffled_metrics.roc_auc < 0.57 else
      "\nINVESTIGATE — a label-free model still ranks better than chance. "
      "Look for duplicate rows or an ID-like feature.")

## Step 9 — Write the methods paragraph

From the lesson:

- **Bad:** "Model achieved 99% accuracy."
- **Better:** "Model was evaluated on held-out test data with train-only preprocessing."
- **Best:** "Error analysis and leakage checks were documented."

Fill this in with your own numbers and save it. This paragraph is what makes every earlier lab believable.

In [ ]:
methods = f"""# Lab 07 — Methods & Leakage Audit

## Methods paragraph (this is the part that goes in the write-up)
The model was trained on the official UNSW-NB15 prepared training partition
({len(X_train)} rows) and evaluated on the held-out testing partition
({len(X_test)} rows); no random re-splitting was performed. Both target columns
(`label`, `attack_cat`) were removed from the feature matrix and this was
asserted programmatically. All preprocessing — imputation, scaling and one-hot
encoding — was fitted on training data only and applied to the test set through
a scikit-learn Pipeline, so no test-set statistics influenced training. Exact
duplicate rows shared across the split were counted ({n_overlap},
{n_overlap / len(X_test):.2%} of the test set) and metrics were recalculated
with them removed. A shuffled-label control was run as a sanity check: ROC AUC
fell from {real_metrics.roc_auc:.3f} to {shuffled_metrics.roc_auc:.3f}
(0.50 = chance), indicating the measured signal is not an artifact.

## Leakage checks performed
- target columns removed from X ......... asserted
- preprocessing fitted on train only .... via Pipeline
- exact duplicate overlap ............... {n_overlap} test rows
- approximate entity overlap ............ {n_near} test rows (proxy only)
- shuffled-label control ................ AUC {shuffled_metrics.roc_auc:.3f}

## Leakage risks I could NOT rule out
1. No session or host identifier exists, so entity-level leakage across the
   split cannot be excluded.
2.

## What this validation cannot prove
The dataset carries no timestamps, so train-on-past / test-on-future was not
possible. These results therefore say nothing about how the model would hold up
against traffic that drifts over time.

## Reported headline metric (after correction)
F1 {reported.f1:.4f} | recall {reported.recall:.4f} | FNR {reported.false_negative_rate:.4f}
"""

methods_path = RESULTS_DIR / "lab07_methods_and_leakage_audit.md"
methods_path.write_text(methods)
print("Saved:", methods_path)
print(methods)

## Your analysis

1. How large was the gap between the honest and the leaky preprocessing? Does a small gap make the shortcut acceptable — and why is that the wrong question?
2. What did the duplicate check find, and how would you report a result of zero?
3. Did the shuffled-label model collapse? If it scored above chance, name your first suspect.
4. This dataset ships with no timestamps, so train-on-past / test-on-future is not available here. What would you need to add, and what would that change about your claims?
5. Rewrite one sentence from an earlier lab's conclusion so that it survives a leakage-aware reader.

**Deliverables:** target-column assertion, train-only preprocessing, a `Pipeline`, a duplicate-overlap check, a shuffled-label control, and written validation notes with the leakage risks you could not rule out.